# Session 1 · Part 1 — Build the tutorial data object

**Goal:** load paired spatial RNA and protein measurements into one validated object. We use the
lightweight `SpatialOmicsData` object. The
object plays the same teaching role: it keeps observations, modalities, and coordinates aligned.

By the end you should be able to explain why row identity and coordinate validation must happen
before normalization or graph construction. Every tutorial section uses the same official Breast
RNA/ADT pair downloaded during Session 0.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Load all three linked tables


In [ ]:
import numpy as np
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.processing import validate_modalities

pair = find_dgat_h5ad_pair(paths.raw_data)
dataset = load_tutorial_data(paths.raw_data)
spots = dataset.spots.copy()
transcripts = dataset.transcripts.select_dtypes(include=[np.number]).copy()
proteins = dataset.proteins.select_dtypes(include=[np.number]).copy()
source = f"RNA={pair[0]}, ADT={pair[1]}"

print(type(dataset).__name__)
print(f"Source: {source}")
print(f"spots × genes: {transcripts.shape}; spots × proteins: {proteins.shape}")
display(spots.head(3), transcripts.iloc[:3, :5], proteins.iloc[:3, :5])


## 2. Validate the object

DGAT assumes that row *i* in RNA, protein, and spatial coordinates is the same biological spot.
The next call checks ordered IDs, duplicate IDs, coordinates, finite values, and non-negative abundance.
A shape match alone is not sufficient.


In [ ]:
validate_modalities(spots, transcripts, proteins)
checks = pd.DataFrame({
    "check": ["ordered IDs match", "x/y coordinates", "finite RNA", "finite protein", "non-negative values"],
    "passed": [
        spots.index.equals(transcripts.index) and spots.index.equals(proteins.index),
        {"x", "y"}.issubset(spots.columns),
        np.isfinite(transcripts.to_numpy()).all(),
        np.isfinite(proteins.to_numpy()).all(),
        (transcripts.to_numpy() >= 0).all() and (proteins.to_numpy() >= 0).all(),
    ],
})
checks


## 3. Record a reproducible input summary


In [ ]:
summary = pd.DataFrame([{
    "source": source,
    "spots": len(spots),
    "genes": transcripts.shape[1],
    "proteins": proteins.shape[1],
    "has_xy_coordinates": {"x", "y"}.issubset(spots.columns),
    "all_validation_checks_passed": bool(checks["passed"].all()),
}])
summary_path = paths.results / "session01_dataset_summary.csv"
summary.to_csv(summary_path, index=False)
manifest = write_checkpoint("1.1", [summary_path], summary=summary.iloc[0].to_dict(), start=paths.root)
summary


## Check

Continue only when every validation row is `True`. If IDs disagree, fix the upstream pairing—never
sort modalities independently and hope they align.
